# Polars: practice notebook

11 exercises, Easy to Hard, each with hidden tests.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("p1")` prints the reference solution for that exercise.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, pandas, scipy, scikit-learn and matplotlib)
import importlib.util, subprocess, sys
for _p in ['polars']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The datasets every exercise uses. Read them if you like; don't change them.

In [ ]:
import polars as pl
import polars.selectors as cs
import numpy as np
from datetime import date, datetime, timedelta
from polars.testing import assert_frame_equal
_rng = np.random.default_rng(42)
_C = ["CH", "DE", "FR", "UK"]
_n = 400
orders = pl.DataFrame({
    "order_id": np.arange(1, _n + 1),
    "user_id": _rng.integers(1, 61, _n),
    "date": [date(2026, 1, 1) + timedelta(days=int(d)) for d in _rng.integers(0, 30, _n)],
    "country": _rng.choice(_C, _n).tolist(),
    "amount": np.round(_rng.lognormal(3, 0.6, _n), 2),
})
_uid = list(range(1, 56)) + list(range(70, 75))
users = pl.DataFrame({
    "user_id": _uid,
    "age": [None if i % 9 == 0 else int(a) for i, a in enumerate(_rng.integers(18, 70, 60))],
    "country": [None if i % 13 == 5 else c for i, c in enumerate(_rng.choice(_C, 60).tolist())],
}, schema={"user_id": pl.Int64, "age": pl.Int64, "country": pl.String})
features = pl.DataFrame({
    "id": np.arange(1, 101),
    "spread": np.round(_rng.normal(0.02, 0.005, 100), 5),
    "volume": np.round(_rng.lognormal(8, 1, 100), 1),
    "trades": _rng.integers(1, 300, 100),
    "momentum": np.round(_rng.normal(0, 1, 100), 4),
    "venue": _rng.choice(["A", "B"], 100).tolist(),
})
_t0 = datetime(2026, 3, 2, 9, 30)
_m = 900
_secs = np.sort(_rng.integers(0, 3600, _m))
ticks = pl.DataFrame({
    "ts": [_t0 + timedelta(seconds=int(s)) for s in _secs],
    "price": np.round(100 + np.cumsum(_rng.normal(0, 0.05, _m)), 2),
    "size": _rng.integers(1, 500, _m),
})
def _book(sym, base, nq, nt):
    qs = np.sort(_rng.integers(0, 1800, nq)); ts_ = np.sort(_rng.integers(20, 1800, nt))
    mid = np.round(base + np.cumsum(_rng.normal(0, 0.02, nq)), 3)
    q = pl.DataFrame({"ts": [_t0 + timedelta(seconds=int(s)) for s in qs], "symbol": [sym] * nq,
                      "bid": np.round(mid - 0.01, 3), "ask": np.round(mid + 0.01, 3)})
    t = pl.DataFrame({"ts": [_t0 + timedelta(seconds=int(s)) for s in ts_], "symbol": [sym] * nt,
                      "price": np.round(base + _rng.normal(0, 0.05, nt), 2), "size": _rng.integers(1, 200, nt)})
    return q, t
_qa, _ta = _book("AAPL", 230.0, 300, 150)
_qb, _tb = _book("MSFT", 410.0, 300, 150)
quotes = pl.concat([_qa, _qb]).sort("ts")
trades = pl.concat([_ta, _tb]).sort("ts")
_days = [date(2026, 1, 1) + timedelta(days=i) for i in range(60)]
prices = pl.DataFrame({
    "symbol": ["AAA"] * 60 + ["BBB"] * 60,
    "date": _days * 2,
    "close": np.round(np.concatenate([50 * np.exp(np.cumsum(_rng.normal(0, 0.02, 60))),
                                      80 * np.exp(np.cumsum(_rng.normal(0, 0.02, 60)))]), 2),
})
PATH = "/tmp/orders.csv"
orders.write_csv(PATH)
del _rng

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJwMSI6IHsic29sdXRpb24iOiAiZGVmIHNvbHZlKG9yZGVycyk6XG4gICAgcmV0dXJuIChcbiAgICAgICAgb3JkZXJzLmdyb3VwX2J5KFwiY291bnRyeVwiKVxuICAgICAgICAuYWdnKFxuICAgICAgICAgICAgbl9vcmRlcnM9cGwubGVuKCksXG4gICAgICAgICAgICB0b3RhbD1wbC5jb2woXCJhbW91bnRcIikuc3VtKCkucm91bmQoMiksXG4gICAgICAgICAgICBhdmdfYW1vdW50PXBsLmNvbChcImFtb3VudFwiKS5tZWFuKCkucm91bmQoMiksXG4gICAgICAgIClcbiAgICAgICAgLnNvcnQoXCJ0b3RhbFwiLCBkZXNjZW5kaW5nPVRydWUpXG4gICAgKVxuIiwgInRlc3RzIjogIlxuZGVmIHNhbWUoYSwgYiwgbXNnPVwiXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInNvbHZlKCkgcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGlzaW5zdGFuY2UoYSwgcGwuRGF0YUZyYW1lKSwgZlwiZXhwZWN0ZWQgYSBwb2xhcnMgRGF0YUZyYW1lLCBnb3Qge3R5cGUoYSkuX19uYW1lX199XCJcbiAgICBhc3NlcnQgYS5jb2x1bW5zID09IGIuY29sdW1ucywgZlwiY29sdW1ucyBhcmUge2EuY29sdW1uc30sIGV4cGVjdGVkIHtiLmNvbHVtbnN9XCJcbiAgICBhc3NlcnQgYS5oZWlnaHQgPT0gYi5oZWlnaHQsIGZcInthLmhlaWdodH0gcm93cywgZXhwZWN0ZWQge2IuaGVpZ2h0fVwiXG4gICAgdHJ5OlxuICAgICAgICBhc3NlcnRfZnJhbWVfZXF1YWwoYSwgYiwgY2hlY2tfZHR5cGVzPUZhbHNlKVxuICAgIGV4Y2VwdCBBc3NlcnRpb25FcnJvciBhcyBlOlxuICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigobXNnICsgXCIgXCIgaWYgbXNnIGVsc2UgXCJcIikgKyBcInZhbHVlcyBkaWZmZXI6IFwiICsgc3RyKGUpLnNwbGl0KFwiXFxuXCIpWzBdKVxuXG5URVNUUyA9IFtcbiAgKFwiZnVsbCBkYXRhc2V0XCIsIGxhbWJkYTogc2FtZShVW1wic29sdmVcIl0ob3JkZXJzKSwgUltcInNvbHZlXCJdKG9yZGVycykpKSxcbiAgKFwiaGlkZGVuIHN1YnNldFwiLCBsYW1iZGE6IHNhbWUoVVtcInNvbHZlXCJdKG9yZGVycy5maWx0ZXIocGwuY29sKFwiYW1vdW50XCIpID4gMjUpKSwgUltcInNvbHZlXCJdKG9yZGVycy5maWx0ZXIocGwuY29sKFwiYW1vdW50XCIpID4gMjUpKSkpLFxuXSIsICJsYW5nIjogInB5In0sICJwMiI6IHsic29sdXRpb24iOiAiZGVmIHNvbHZlKHVzZXJzKTpcbiAgICByZXR1cm4gKFxuICAgICAgICB1c2Vycy5kcm9wX251bGxzKFwiY291bnRyeVwiKVxuICAgICAgICAud2l0aF9jb2x1bW5zKFxuICAgICAgICAgICAgcGwuY29sKFwiYWdlXCIpLmZpbGxfbnVsbChcbiAgICAgICAgICAgICAgICBwbC5jb2woXCJhZ2VcIikubWVkaWFuKCkub3ZlcihcImNvdW50cnlcIikucm91bmQoMCkuY2FzdChwbC5JbnQ2NClcbiAgICAgICAgICAgIClcbiAgICAgICAgKVxuICAgICAgICAuc29ydChcInVzZXJfaWRcIilcbiAgICApXG4iLCAidGVzdHMiOiAiXG5kZWYgc2FtZShhLCBiLCBtc2c9XCJcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwic29sdmUoKSByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBwbC5EYXRhRnJhbWUpLCBmXCJleHBlY3RlZCBhIHBvbGFycyBEYXRhRnJhbWUsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBhLmNvbHVtbnMgPT0gYi5jb2x1bW5zLCBmXCJjb2x1bW5zIGFyZSB7YS5jb2x1bW5zfSwgZXhwZWN0ZWQge2IuY29sdW1uc31cIlxuICAgIGFzc2VydCBhLmhlaWdodCA9PSBiLmhlaWdodCwgZlwie2EuaGVpZ2h0fSByb3dzLCBleHBlY3RlZCB7Yi5oZWlnaHR9XCJcbiAgICB0cnk6XG4gICAgICAgIGFzc2VydF9mcmFtZV9lcXVhbChhLCBiLCBjaGVja19kdHlwZXM9RmFsc2UpXG4gICAgZXhjZXB0IEFzc2VydGlvbkVycm9yIGFzIGU6XG4gICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKChtc2cgKyBcIiBcIiBpZiBtc2cgZWxzZSBcIlwiKSArIFwidmFsdWVzIGRpZmZlcjogXCIgKyBzdHIoZSkuc3BsaXQoXCJcXG5cIilbMF0pXG5cbmRlZiBfbm9udWxsKCk6XG4gICAgciA9IFVbXCJzb2x2ZVwiXSh1c2VycylcbiAgICBhc3NlcnQgciBpcyBub3QgTm9uZSwgXCJzb2x2ZSgpIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGFzc2VydCByLm51bGxfY291bnQoKS5zdW1faG9yaXpvbnRhbCgpWzBdID09IDAsIFwidGhlIHJlc3VsdCBzdGlsbCBoYXMgbnVsbHNcIlxuVEVTVFMgPSBbXG4gIChcIm5vIG51bGxzIGxlZnRcIiwgX25vbnVsbCksXG4gIChcImZ1bGwgZGF0YXNldFwiLCBsYW1iZGE6IHNhbWUoVVtcInNvbHZlXCJdKHVzZXJzKSwgUltcInNvbHZlXCJdKHVzZXJzKSkpLFxuXSIsICJsYW5nIjogInB5In0sICJwMyI6IHsic29sdXRpb24iOiAiZGVmIHNvbHZlKGZlYXR1cmVzKTpcbiAgICB4ID0gcGwuY29sKHBsLkZsb2F0NjQpXG4gICAgcmV0dXJuIGZlYXR1cmVzLndpdGhfY29sdW1ucygoKHggLSB4Lm1lYW4oKSkgLyB4LnN0ZCgpKS5uYW1lLnN1ZmZpeChcIl96XCIpKVxuIiwgInRlc3RzIjogIlxuZGVmIHNhbWUoYSwgYiwgbXNnPVwiXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInNvbHZlKCkgcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGlzaW5zdGFuY2UoYSwgcGwuRGF0YUZyYW1lKSwgZlwiZXhwZWN0ZWQgYSBwb2xhcnMgRGF0YUZyYW1lLCBnb3Qge3R5cGUoYSkuX19uYW1lX199XCJcbiAgICBhc3NlcnQgYS5jb2x1bW5zID09IGIuY29sdW1ucywgZlwiY29sdW1ucyBhcmUge2EuY29sdW1uc30sIGV4cGVjdGVkIHtiLmNvbHVtbnN9XCJcbiAgICBhc3NlcnQgYS5oZWlnaHQgPT0gYi5oZWlnaHQsIGZcInthLmhlaWdodH0gcm93cywgZXhwZWN0ZWQge2IuaGVpZ2h0fVwiXG4gICAgdHJ5OlxuICAgICAgICBhc3NlcnRfZnJhbWVfZXF1YWwoYSwgYiwgY2hlY2tfZHR5cGVzPUZhbHNlKVxuICAgIGV4Y2VwdCBBc3NlcnRpb25FcnJvciBhcyBlOlxuICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigobXNnICsgXCIgXCIgaWYgbXNnIGVsc2UgXCJcIikgKyBcInZhbHVlcyBkaWZmZXI6IFwiICsgc3RyKGUpLnNwbGl0KFwiXFxuXCIpWzBdKVxuXG5kZWYgX3JlZihkZik6XG4gICAgZmwgPSBbYyBmb3IgYywgdCBpbiBkZi5zY2hlbWEuaXRlbXMoKSBpZiB0ID09IHBsLkZsb2F0NjRdXG4gICAgcmV0dXJuIGRmLndpdGhfY29sdW1ucyhbKChwbC5jb2woYykgLSBwbC5jb2woYykubWVhbigpKSAvIHBsLmNvbChjKS5zdGQoKSkuYWxpYXMoYyArIFwiX3pcIikgZm9yIGMgaW4gZmxdKVxuX2V4dHJhID0gZmVhdHVyZXMud2l0aF9jb2x1bW5zKGJldGE9KHBsLmNvbChcIm1vbWVudHVtXCIpICogMC4zICsgMS4wKSlcblRFU1RTID0gW1xuICAoXCJmdWxsIGRhdGFzZXRcIiwgbGFtYmRhOiBzYW1lKFVbXCJzb2x2ZVwiXShmZWF0dXJlcyksIF9yZWYoZmVhdHVyZXMpKSksXG4gIChcIndvcmtzIG9uIGEgbmV3IGZsb2F0IGNvbHVtblwiLCBsYW1iZGE6IHNhbWUoVVtcInNvbHZlXCJdKF9leHRyYSksIF9yZWYoX2V4dHJhKSwgXCJhIG5ldyBmbG9hdCBjb2x1bW4gJ2JldGEnIHdhcyBhZGRlZDpcIikpLFxuXSIsICJsYW5nIjogInB5In0sICJwNCI6IHsic29sdXRpb24iOiAiZGVmIHNvbHZlKG9yZGVycywgdXNlcnMpOlxuICAgIG9ycGhhbnMgPSBvcmRlcnMuam9pbih1c2Vycywgb249XCJ1c2VyX2lkXCIsIGhvdz1cImFudGlcIikuc29ydChcIm9yZGVyX2lkXCIpXG4gICAgaW5hY3RpdmUgPSB1c2Vycy5qb2luKG9yZGVycywgb249XCJ1c2VyX2lkXCIsIGhvdz1cImFudGlcIikuc29ydChcInVzZXJfaWRcIilcbiAgICByZXR1cm4gb3JwaGFucywgaW5hY3RpdmVcbiIsICJ0ZXN0cyI6ICJcbmRlZiBzYW1lKGEsIGIsIG1zZz1cIlwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJzb2x2ZSgpIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIHBsLkRhdGFGcmFtZSksIGZcImV4cGVjdGVkIGEgcG9sYXJzIERhdGFGcmFtZSwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IGEuY29sdW1ucyA9PSBiLmNvbHVtbnMsIGZcImNvbHVtbnMgYXJlIHthLmNvbHVtbnN9LCBleHBlY3RlZCB7Yi5jb2x1bW5zfVwiXG4gICAgYXNzZXJ0IGEuaGVpZ2h0ID09IGIuaGVpZ2h0LCBmXCJ7YS5oZWlnaHR9IHJvd3MsIGV4cGVjdGVkIHtiLmhlaWdodH1cIlxuICAgIHRyeTpcbiAgICAgICAgYXNzZXJ0X2ZyYW1lX2VxdWFsKGEsIGIsIGNoZWNrX2R0eXBlcz1GYWxzZSlcbiAgICBleGNlcHQgQXNzZXJ0aW9uRXJyb3IgYXMgZTpcbiAgICAgICAgcmFpc2UgQXNzZXJ0aW9uRXJyb3IoKG1zZyArIFwiIFwiIGlmIG1zZyBlbHNlIFwiXCIpICsgXCJ2YWx1ZXMgZGlmZmVyOiBcIiArIHN0cihlKS5zcGxpdChcIlxcblwiKVswXSlcblxuZGVmIF90KGspOlxuICAgIGEgPSBVW1wic29sdmVcIl0ob3JkZXJzLCB1c2Vycyk7IGIgPSBSW1wic29sdmVcIl0ob3JkZXJzLCB1c2VycylcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCB0dXBsZSkgYW5kIGxlbihhKSA9PSAyLCBcInJldHVybiBhIHR1cGxlIChvcnBoYW5zLCBpbmFjdGl2ZSlcIlxuICAgIHNhbWUoYVtrXSwgYltrXSwgW1wib3JwaGFuczpcIiwgXCJpbmFjdGl2ZTpcIl1ba10pXG5URVNUUyA9IFsoXCJvcnBoYW4gb3JkZXJzXCIsIGxhbWJkYTogX3QoMCkpLCAoXCJpbmFjdGl2ZSB1c2Vyc1wiLCBsYW1iZGE6IF90KDEpKV0iLCAibGFuZyI6ICJweSJ9LCAicDUiOiB7InNvbHV0aW9uIjogImRlZiBzb2x2ZShvcmRlcnMpOlxuICAgIGRhaWx5ID0gb3JkZXJzLmdyb3VwX2J5KFwiZGF0ZVwiLCBcImNvdW50cnlcIikuYWdnKHBsLmNvbChcImFtb3VudFwiKS5zdW0oKSlcbiAgICByZXR1cm4gKFxuICAgICAgICBkYWlseS5waXZvdChvbj1cImNvdW50cnlcIiwgaW5kZXg9XCJkYXRlXCIsIHZhbHVlcz1cImFtb3VudFwiLCBzb3J0X2NvbHVtbnM9VHJ1ZSlcbiAgICAgICAgLnNvcnQoXCJkYXRlXCIpXG4gICAgICAgIC53aXRoX2NvbHVtbnMocGwuY29sKHBsLkZsb2F0NjQpLmZpbGxfbnVsbCgwKS5yb3VuZCgyKSlcbiAgICApXG4iLCAidGVzdHMiOiAiXG5kZWYgc2FtZShhLCBiLCBtc2c9XCJcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwic29sdmUoKSByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBwbC5EYXRhRnJhbWUpLCBmXCJleHBlY3RlZCBhIHBvbGFycyBEYXRhRnJhbWUsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBhLmNvbHVtbnMgPT0gYi5jb2x1bW5zLCBmXCJjb2x1bW5zIGFyZSB7YS5jb2x1bW5zfSwgZXhwZWN0ZWQge2IuY29sdW1uc31cIlxuICAgIGFzc2VydCBhLmhlaWdodCA9PSBiLmhlaWdodCwgZlwie2EuaGVpZ2h0fSByb3dzLCBleHBlY3RlZCB7Yi5oZWlnaHR9XCJcbiAgICB0cnk6XG4gICAgICAgIGFzc2VydF9mcmFtZV9lcXVhbChhLCBiLCBjaGVja19kdHlwZXM9RmFsc2UpXG4gICAgZXhjZXB0IEFzc2VydGlvbkVycm9yIGFzIGU6XG4gICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKChtc2cgKyBcIiBcIiBpZiBtc2cgZWxzZSBcIlwiKSArIFwidmFsdWVzIGRpZmZlcjogXCIgKyBzdHIoZSkuc3BsaXQoXCJcXG5cIilbMF0pXG5cbl9zdWIgPSBvcmRlcnMuZmlsdGVyKHBsLmNvbChcImNvdW50cnlcIikgIT0gXCJGUlwiKVxuVEVTVFMgPSBbXG4gIChcImZ1bGwgZGF0YXNldFwiLCBsYW1iZGE6IHNhbWUoVVtcInNvbHZlXCJdKG9yZGVycyksIFJbXCJzb2x2ZVwiXShvcmRlcnMpKSksXG4gIChcImhpZGRlbjogYSBjb3VudHJ5IG1pc3NpbmdcIiwgbGFtYmRhOiBzYW1lKFVbXCJzb2x2ZVwiXShfc3ViKSwgUltcInNvbHZlXCJdKF9zdWIpKSksXG5dIiwgImxhbmciOiAicHkifSwgInA2IjogeyJzb2x1dGlvbiI6ICJkZWYgc29sdmUob3JkZXJzKTpcbiAgICByZXR1cm4gKFxuICAgICAgICBvcmRlcnMuc29ydChcInVzZXJfaWRcIiwgXCJkYXRlXCIsIFwib3JkZXJfaWRcIilcbiAgICAgICAgLndpdGhfY29sdW1ucyhcbiAgICAgICAgICAgIHVzZXJfdG90YWw9cGwuY29sKFwiYW1vdW50XCIpLnN1bSgpLm92ZXIoXCJ1c2VyX2lkXCIpLFxuICAgICAgICAgICAgY3VtX3NwZW5kPXBsLmNvbChcImFtb3VudFwiKS5jdW1fc3VtKCkub3ZlcihcInVzZXJfaWRcIiksXG4gICAgICAgIClcbiAgICAgICAgLndpdGhfY29sdW1ucyhzaGFyZT0ocGwuY29sKFwiYW1vdW50XCIpIC8gcGwuY29sKFwidXNlcl90b3RhbFwiKSkucm91bmQoNCkpXG4gICAgKVxuIiwgInRlc3RzIjogIlxuZGVmIHNhbWUoYSwgYiwgbXNnPVwiXCIpOlxuICAgIGFzc2VydCBhIGlzIG5vdCBOb25lLCBcInNvbHZlKCkgcmV0dXJuZWQgTm9uZS4gRGlkIHlvdSBmb3JnZXQgdG8gcmV0dXJuP1wiXG4gICAgYXNzZXJ0IGlzaW5zdGFuY2UoYSwgcGwuRGF0YUZyYW1lKSwgZlwiZXhwZWN0ZWQgYSBwb2xhcnMgRGF0YUZyYW1lLCBnb3Qge3R5cGUoYSkuX19uYW1lX199XCJcbiAgICBhc3NlcnQgYS5jb2x1bW5zID09IGIuY29sdW1ucywgZlwiY29sdW1ucyBhcmUge2EuY29sdW1uc30sIGV4cGVjdGVkIHtiLmNvbHVtbnN9XCJcbiAgICBhc3NlcnQgYS5oZWlnaHQgPT0gYi5oZWlnaHQsIGZcInthLmhlaWdodH0gcm93cywgZXhwZWN0ZWQge2IuaGVpZ2h0fVwiXG4gICAgdHJ5OlxuICAgICAgICBhc3NlcnRfZnJhbWVfZXF1YWwoYSwgYiwgY2hlY2tfZHR5cGVzPUZhbHNlKVxuICAgIGV4Y2VwdCBBc3NlcnRpb25FcnJvciBhcyBlOlxuICAgICAgICByYWlzZSBBc3NlcnRpb25FcnJvcigobXNnICsgXCIgXCIgaWYgbXNnIGVsc2UgXCJcIikgKyBcInZhbHVlcyBkaWZmZXI6IFwiICsgc3RyKGUpLnNwbGl0KFwiXFxuXCIpWzBdKVxuXG5URVNUUyA9IFsoXCJmdWxsIGRhdGFzZXRcIiwgbGFtYmRhOiBzYW1lKFVbXCJzb2x2ZVwiXShvcmRlcnMpLCBSW1wic29sdmVcIl0ob3JkZXJzKSkpXSIsICJsYW5nIjogInB5In0sICJwNyI6IHsic29sdXRpb24iOiAiZGVmIHNvbHZlKG9yZGVycyk6XG4gICAgcmV0dXJuIChcbiAgICAgICAgb3JkZXJzLmdyb3VwX2J5KFwiY291bnRyeVwiLCBcImRhdGVcIilcbiAgICAgICAgLmFnZyhyZXZlbnVlPXBsLmNvbChcImFtb3VudFwiKS5zdW0oKSlcbiAgICAgICAgLnNvcnQoXCJjb3VudHJ5XCIsIFwiZGF0ZVwiKVxuICAgICAgICAud2l0aF9jb2x1bW5zKHJldl8zZD1wbC5jb2woXCJyZXZlbnVlXCIpLnJvbGxpbmdfbWVhbigzKS5vdmVyKFwiY291bnRyeVwiKSlcbiAgICApXG4iLCAidGVzdHMiOiAiXG5kZWYgc2FtZShhLCBiLCBtc2c9XCJcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwic29sdmUoKSByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBwbC5EYXRhRnJhbWUpLCBmXCJleHBlY3RlZCBhIHBvbGFycyBEYXRhRnJhbWUsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBhLmNvbHVtbnMgPT0gYi5jb2x1bW5zLCBmXCJjb2x1bW5zIGFyZSB7YS5jb2x1bW5zfSwgZXhwZWN0ZWQge2IuY29sdW1uc31cIlxuICAgIGFzc2VydCBhLmhlaWdodCA9PSBiLmhlaWdodCwgZlwie2EuaGVpZ2h0fSByb3dzLCBleHBlY3RlZCB7Yi5oZWlnaHR9XCJcbiAgICB0cnk6XG4gICAgICAgIGFzc2VydF9mcmFtZV9lcXVhbChhLCBiLCBjaGVja19kdHlwZXM9RmFsc2UpXG4gICAgZXhjZXB0IEFzc2VydGlvbkVycm9yIGFzIGU6XG4gICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKChtc2cgKyBcIiBcIiBpZiBtc2cgZWxzZSBcIlwiKSArIFwidmFsdWVzIGRpZmZlcjogXCIgKyBzdHIoZSkuc3BsaXQoXCJcXG5cIilbMF0pXG5cblRFU1RTID0gWyhcImZ1bGwgZGF0YXNldFwiLCBsYW1iZGE6IHNhbWUoVVtcInNvbHZlXCJdKG9yZGVycyksIFJbXCJzb2x2ZVwiXShvcmRlcnMpKSldIiwgImxhbmciOiAicHkifSwgInA4IjogeyJzb2x1dGlvbiI6ICJkZWYgc29sdmUodGlja3MpOlxuICAgIHJldHVybiB0aWNrcy5zb3J0KFwidHNcIikuZ3JvdXBfYnlfZHluYW1pYyhcInRzXCIsIGV2ZXJ5PVwiMW1cIikuYWdnKFxuICAgICAgICBvcGVuPXBsLmNvbChcInByaWNlXCIpLmZpcnN0KCksXG4gICAgICAgIGhpZ2g9cGwuY29sKFwicHJpY2VcIikubWF4KCksXG4gICAgICAgIGxvdz1wbC5jb2woXCJwcmljZVwiKS5taW4oKSxcbiAgICAgICAgY2xvc2U9cGwuY29sKFwicHJpY2VcIikubGFzdCgpLFxuICAgICAgICB2b2x1bWU9cGwuY29sKFwic2l6ZVwiKS5zdW0oKSxcbiAgICApXG4iLCAidGVzdHMiOiAiXG5kZWYgc2FtZShhLCBiLCBtc2c9XCJcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwic29sdmUoKSByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBwbC5EYXRhRnJhbWUpLCBmXCJleHBlY3RlZCBhIHBvbGFycyBEYXRhRnJhbWUsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBhLmNvbHVtbnMgPT0gYi5jb2x1bW5zLCBmXCJjb2x1bW5zIGFyZSB7YS5jb2x1bW5zfSwgZXhwZWN0ZWQge2IuY29sdW1uc31cIlxuICAgIGFzc2VydCBhLmhlaWdodCA9PSBiLmhlaWdodCwgZlwie2EuaGVpZ2h0fSByb3dzLCBleHBlY3RlZCB7Yi5oZWlnaHR9XCJcbiAgICB0cnk6XG4gICAgICAgIGFzc2VydF9mcmFtZV9lcXVhbChhLCBiLCBjaGVja19kdHlwZXM9RmFsc2UpXG4gICAgZXhjZXB0IEFzc2VydGlvbkVycm9yIGFzIGU6XG4gICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKChtc2cgKyBcIiBcIiBpZiBtc2cgZWxzZSBcIlwiKSArIFwidmFsdWVzIGRpZmZlcjogXCIgKyBzdHIoZSkuc3BsaXQoXCJcXG5cIilbMF0pXG5cbl9oYWxmID0gdGlja3MuZmlsdGVyKHBsLmNvbChcInRzXCIpLmR0Lm1pbnV0ZSgpICUgMiA9PSAwKVxuVEVTVFMgPSBbXG4gIChcImZ1bGwgaG91clwiLCBsYW1iZGE6IHNhbWUoVVtcInNvbHZlXCJdKHRpY2tzKSwgUltcInNvbHZlXCJdKHRpY2tzKSkpLFxuICAoXCJoaWRkZW46IGdhcHMgaW4gdGhlIGRhdGFcIiwgbGFtYmRhOiBzYW1lKFVbXCJzb2x2ZVwiXShfaGFsZiksIFJbXCJzb2x2ZVwiXShfaGFsZikpKSxcbl0iLCAibGFuZyI6ICJweSJ9LCAicDkiOiB7InNvbHV0aW9uIjogImRlZiBzb2x2ZSh0cmFkZXMsIHF1b3Rlcyk6XG4gICAgcmV0dXJuIChcbiAgICAgICAgdHJhZGVzLnNvcnQoXCJ0c1wiKVxuICAgICAgICAuam9pbl9hc29mKHF1b3Rlcy5zb3J0KFwidHNcIiksIG9uPVwidHNcIiwgYnk9XCJzeW1ib2xcIiwgc3RyYXRlZ3k9XCJiYWNrd2FyZFwiKVxuICAgICAgICAud2l0aF9jb2x1bW5zKG1pZD0ocGwuY29sKFwiYmlkXCIpICsgcGwuY29sKFwiYXNrXCIpKSAvIDIpXG4gICAgICAgIC53aXRoX2NvbHVtbnMoYnV5ZXJfaW5pdGlhdGVkPXBsLmNvbChcInByaWNlXCIpID49IHBsLmNvbChcIm1pZFwiKSlcbiAgICAgICAgLnNlbGVjdChcInRzXCIsIFwic3ltYm9sXCIsIFwicHJpY2VcIiwgXCJzaXplXCIsIFwiYmlkXCIsIFwiYXNrXCIsIFwibWlkXCIsIFwiYnV5ZXJfaW5pdGlhdGVkXCIpXG4gICAgICAgIC5zb3J0KFwic3ltYm9sXCIsIFwidHNcIilcbiAgICApXG4iLCAidGVzdHMiOiAiXG5kZWYgc2FtZShhLCBiLCBtc2c9XCJcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwic29sdmUoKSByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBwbC5EYXRhRnJhbWUpLCBmXCJleHBlY3RlZCBhIHBvbGFycyBEYXRhRnJhbWUsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBhLmNvbHVtbnMgPT0gYi5jb2x1bW5zLCBmXCJjb2x1bW5zIGFyZSB7YS5jb2x1bW5zfSwgZXhwZWN0ZWQge2IuY29sdW1uc31cIlxuICAgIGFzc2VydCBhLmhlaWdodCA9PSBiLmhlaWdodCwgZlwie2EuaGVpZ2h0fSByb3dzLCBleHBlY3RlZCB7Yi5oZWlnaHR9XCJcbiAgICB0cnk6XG4gICAgICAgIGFzc2VydF9mcmFtZV9lcXVhbChhLCBiLCBjaGVja19kdHlwZXM9RmFsc2UpXG4gICAgZXhjZXB0IEFzc2VydGlvbkVycm9yIGFzIGU6XG4gICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKChtc2cgKyBcIiBcIiBpZiBtc2cgZWxzZSBcIlwiKSArIFwidmFsdWVzIGRpZmZlcjogXCIgKyBzdHIoZSkuc3BsaXQoXCJcXG5cIilbMF0pXG5cblRFU1RTID0gWyhcImZ1bGwgZGF0YXNldFwiLCBsYW1iZGE6IHNhbWUoVVtcInNvbHZlXCJdKHRyYWRlcywgcXVvdGVzKSwgUltcInNvbHZlXCJdKHRyYWRlcywgcXVvdGVzKSkpXSIsICJsYW5nIjogInB5In0sICJwMTAiOiB7InNvbHV0aW9uIjogImRlZiBzb2x2ZShwcmljZXMpOlxuICAgIHJldHVybiAoXG4gICAgICAgIHByaWNlcy5zb3J0KFwic3ltYm9sXCIsIFwiZGF0ZVwiKVxuICAgICAgICAud2l0aF9jb2x1bW5zKFxuICAgICAgICAgICAgcmV0PXBsLmNvbChcImNsb3NlXCIpLnBjdF9jaGFuZ2UoKS5vdmVyKFwic3ltYm9sXCIpLFxuICAgICAgICAgICAgc2lnbmFsPShwbC5jb2woXCJjbG9zZVwiKSA+IHBsLmNvbChcImNsb3NlXCIpLnJvbGxpbmdfbWVhbig1KS5vdmVyKFwic3ltYm9sXCIpKS5jYXN0KHBsLkludDgpICogMiAtIDEsXG4gICAgICAgIClcbiAgICAgICAgLndpdGhfY29sdW1ucyhzdHJhdGVneV9yZXQ9KHBsLmNvbChcInNpZ25hbFwiKS5zaGlmdCgxKSAqIHBsLmNvbChcInJldFwiKSkub3ZlcihcInN5bWJvbFwiKSlcbiAgICAgICAgLnNlbGVjdChcInN5bWJvbFwiLCBcImRhdGVcIiwgXCJyZXRcIiwgXCJzaWduYWxcIiwgXCJzdHJhdGVneV9yZXRcIilcbiAgICApXG4iLCAidGVzdHMiOiAiXG5kZWYgc2FtZShhLCBiLCBtc2c9XCJcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwic29sdmUoKSByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBwbC5EYXRhRnJhbWUpLCBmXCJleHBlY3RlZCBhIHBvbGFycyBEYXRhRnJhbWUsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBhLmNvbHVtbnMgPT0gYi5jb2x1bW5zLCBmXCJjb2x1bW5zIGFyZSB7YS5jb2x1bW5zfSwgZXhwZWN0ZWQge2IuY29sdW1uc31cIlxuICAgIGFzc2VydCBhLmhlaWdodCA9PSBiLmhlaWdodCwgZlwie2EuaGVpZ2h0fSByb3dzLCBleHBlY3RlZCB7Yi5oZWlnaHR9XCJcbiAgICB0cnk6XG4gICAgICAgIGFzc2VydF9mcmFtZV9lcXVhbChhLCBiLCBjaGVja19kdHlwZXM9RmFsc2UpXG4gICAgZXhjZXB0IEFzc2VydGlvbkVycm9yIGFzIGU6XG4gICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKChtc2cgKyBcIiBcIiBpZiBtc2cgZWxzZSBcIlwiKSArIFwidmFsdWVzIGRpZmZlcjogXCIgKyBzdHIoZSkuc3BsaXQoXCJcXG5cIilbMF0pXG5cblRFU1RTID0gWyhcImZ1bGwgZGF0YXNldFwiLCBsYW1iZGE6IHNhbWUoVVtcInNvbHZlXCJdKHByaWNlcyksIFJbXCJzb2x2ZVwiXShwcmljZXMpKSldIiwgImxhbmciOiAicHkifSwgInAxMSI6IHsic29sdXRpb24iOiAiZGVmIHNvbHZlKHBhdGgpOlxuICAgIHJldHVybiAoXG4gICAgICAgIHBsLnNjYW5fY3N2KHBhdGgpXG4gICAgICAgIC5maWx0ZXIocGwuY29sKFwiYW1vdW50XCIpID4gMjApXG4gICAgICAgIC5ncm91cF9ieShcImNvdW50cnlcIilcbiAgICAgICAgLmFnZyhuX29yZGVycz1wbC5sZW4oKSwgdG90YWw9cGwuY29sKFwiYW1vdW50XCIpLnN1bSgpLnJvdW5kKDIpKVxuICAgICAgICAuc29ydChcInRvdGFsXCIsIGRlc2NlbmRpbmc9VHJ1ZSlcbiAgICApXG4iLCAidGVzdHMiOiAiXG5kZWYgc2FtZShhLCBiLCBtc2c9XCJcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwic29sdmUoKSByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShhLCBwbC5EYXRhRnJhbWUpLCBmXCJleHBlY3RlZCBhIHBvbGFycyBEYXRhRnJhbWUsIGdvdCB7dHlwZShhKS5fX25hbWVfX31cIlxuICAgIGFzc2VydCBhLmNvbHVtbnMgPT0gYi5jb2x1bW5zLCBmXCJjb2x1bW5zIGFyZSB7YS5jb2x1bW5zfSwgZXhwZWN0ZWQge2IuY29sdW1uc31cIlxuICAgIGFzc2VydCBhLmhlaWdodCA9PSBiLmhlaWdodCwgZlwie2EuaGVpZ2h0fSByb3dzLCBleHBlY3RlZCB7Yi5oZWlnaHR9XCJcbiAgICB0cnk6XG4gICAgICAgIGFzc2VydF9mcmFtZV9lcXVhbChhLCBiLCBjaGVja19kdHlwZXM9RmFsc2UpXG4gICAgZXhjZXB0IEFzc2VydGlvbkVycm9yIGFzIGU6XG4gICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKChtc2cgKyBcIiBcIiBpZiBtc2cgZWxzZSBcIlwiKSArIFwidmFsdWVzIGRpZmZlcjogXCIgKyBzdHIoZSkuc3BsaXQoXCJcXG5cIilbMF0pXG5cbmRlZiBfbGF6eSgpOlxuICAgIHEgPSBVW1wic29sdmVcIl0oUEFUSClcbiAgICBhc3NlcnQgaXNpbnN0YW5jZShxLCBwbC5MYXp5RnJhbWUpLCBmXCJyZXR1cm4gYSBMYXp5RnJhbWUsIGdvdCB7dHlwZShxKS5fX25hbWVfX31cIlxuVEVTVFMgPSBbXG4gIChcInJldHVybnMgYSBMYXp5RnJhbWVcIiwgX2xhenkpLFxuICAoXCJjb2xsZWN0cyB0byB0aGUgcmlnaHQgYW5zd2VyXCIsIGxhbWJkYTogc2FtZShVW1wic29sdmVcIl0oUEFUSCkuY29sbGVjdCgpLCBSW1wic29sdmVcIl0oUEFUSCkuY29sbGVjdCgpKSksXG5dIiwgImxhbmciOiAicHkifX0=").decode())
_SETUP = _b64.b64decode("CmltcG9ydCBwb2xhcnMgYXMgcGwKaW1wb3J0IHBvbGFycy5zZWxlY3RvcnMgYXMgY3MKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGUsIGRhdGV0aW1lLCB0aW1lZGVsdGEKZnJvbSBwb2xhcnMudGVzdGluZyBpbXBvcnQgYXNzZXJ0X2ZyYW1lX2VxdWFsCl9ybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoNDIpCl9DID0gWyJDSCIsICJERSIsICJGUiIsICJVSyJdCl9uID0gNDAwCm9yZGVycyA9IHBsLkRhdGFGcmFtZSh7CiAgICAib3JkZXJfaWQiOiBucC5hcmFuZ2UoMSwgX24gKyAxKSwKICAgICJ1c2VyX2lkIjogX3JuZy5pbnRlZ2VycygxLCA2MSwgX24pLAogICAgImRhdGUiOiBbZGF0ZSgyMDI2LCAxLCAxKSArIHRpbWVkZWx0YShkYXlzPWludChkKSkgZm9yIGQgaW4gX3JuZy5pbnRlZ2VycygwLCAzMCwgX24pXSwKICAgICJjb3VudHJ5IjogX3JuZy5jaG9pY2UoX0MsIF9uKS50b2xpc3QoKSwKICAgICJhbW91bnQiOiBucC5yb3VuZChfcm5nLmxvZ25vcm1hbCgzLCAwLjYsIF9uKSwgMiksCn0pCl91aWQgPSBsaXN0KHJhbmdlKDEsIDU2KSkgKyBsaXN0KHJhbmdlKDcwLCA3NSkpCnVzZXJzID0gcGwuRGF0YUZyYW1lKHsKICAgICJ1c2VyX2lkIjogX3VpZCwKICAgICJhZ2UiOiBbTm9uZSBpZiBpICUgOSA9PSAwIGVsc2UgaW50KGEpIGZvciBpLCBhIGluIGVudW1lcmF0ZShfcm5nLmludGVnZXJzKDE4LCA3MCwgNjApKV0sCiAgICAiY291bnRyeSI6IFtOb25lIGlmIGkgJSAxMyA9PSA1IGVsc2UgYyBmb3IgaSwgYyBpbiBlbnVtZXJhdGUoX3JuZy5jaG9pY2UoX0MsIDYwKS50b2xpc3QoKSldLAp9LCBzY2hlbWE9eyJ1c2VyX2lkIjogcGwuSW50NjQsICJhZ2UiOiBwbC5JbnQ2NCwgImNvdW50cnkiOiBwbC5TdHJpbmd9KQpmZWF0dXJlcyA9IHBsLkRhdGFGcmFtZSh7CiAgICAiaWQiOiBucC5hcmFuZ2UoMSwgMTAxKSwKICAgICJzcHJlYWQiOiBucC5yb3VuZChfcm5nLm5vcm1hbCgwLjAyLCAwLjAwNSwgMTAwKSwgNSksCiAgICAidm9sdW1lIjogbnAucm91bmQoX3JuZy5sb2dub3JtYWwoOCwgMSwgMTAwKSwgMSksCiAgICAidHJhZGVzIjogX3JuZy5pbnRlZ2VycygxLCAzMDAsIDEwMCksCiAgICAibW9tZW50dW0iOiBucC5yb3VuZChfcm5nLm5vcm1hbCgwLCAxLCAxMDApLCA0KSwKICAgICJ2ZW51ZSI6IF9ybmcuY2hvaWNlKFsiQSIsICJCIl0sIDEwMCkudG9saXN0KCksCn0pCl90MCA9IGRhdGV0aW1lKDIwMjYsIDMsIDIsIDksIDMwKQpfbSA9IDkwMApfc2VjcyA9IG5wLnNvcnQoX3JuZy5pbnRlZ2VycygwLCAzNjAwLCBfbSkpCnRpY2tzID0gcGwuRGF0YUZyYW1lKHsKICAgICJ0cyI6IFtfdDAgKyB0aW1lZGVsdGEoc2Vjb25kcz1pbnQocykpIGZvciBzIGluIF9zZWNzXSwKICAgICJwcmljZSI6IG5wLnJvdW5kKDEwMCArIG5wLmN1bXN1bShfcm5nLm5vcm1hbCgwLCAwLjA1LCBfbSkpLCAyKSwKICAgICJzaXplIjogX3JuZy5pbnRlZ2VycygxLCA1MDAsIF9tKSwKfSkKZGVmIF9ib29rKHN5bSwgYmFzZSwgbnEsIG50KToKICAgIHFzID0gbnAuc29ydChfcm5nLmludGVnZXJzKDAsIDE4MDAsIG5xKSk7IHRzXyA9IG5wLnNvcnQoX3JuZy5pbnRlZ2VycygyMCwgMTgwMCwgbnQpKQogICAgbWlkID0gbnAucm91bmQoYmFzZSArIG5wLmN1bXN1bShfcm5nLm5vcm1hbCgwLCAwLjAyLCBucSkpLCAzKQogICAgcSA9IHBsLkRhdGFGcmFtZSh7InRzIjogW190MCArIHRpbWVkZWx0YShzZWNvbmRzPWludChzKSkgZm9yIHMgaW4gcXNdLCAic3ltYm9sIjogW3N5bV0gKiBucSwKICAgICAgICAgICAgICAgICAgICAgICJiaWQiOiBucC5yb3VuZChtaWQgLSAwLjAxLCAzKSwgImFzayI6IG5wLnJvdW5kKG1pZCArIDAuMDEsIDMpfSkKICAgIHQgPSBwbC5EYXRhRnJhbWUoeyJ0cyI6IFtfdDAgKyB0aW1lZGVsdGEoc2Vjb25kcz1pbnQocykpIGZvciBzIGluIHRzX10sICJzeW1ib2wiOiBbc3ltXSAqIG50LAogICAgICAgICAgICAgICAgICAgICAgInByaWNlIjogbnAucm91bmQoYmFzZSArIF9ybmcubm9ybWFsKDAsIDAuMDUsIG50KSwgMiksICJzaXplIjogX3JuZy5pbnRlZ2VycygxLCAyMDAsIG50KX0pCiAgICByZXR1cm4gcSwgdApfcWEsIF90YSA9IF9ib29rKCJBQVBMIiwgMjMwLjAsIDMwMCwgMTUwKQpfcWIsIF90YiA9IF9ib29rKCJNU0ZUIiwgNDEwLjAsIDMwMCwgMTUwKQpxdW90ZXMgPSBwbC5jb25jYXQoW19xYSwgX3FiXSkuc29ydCgidHMiKQp0cmFkZXMgPSBwbC5jb25jYXQoW190YSwgX3RiXSkuc29ydCgidHMiKQpfZGF5cyA9IFtkYXRlKDIwMjYsIDEsIDEpICsgdGltZWRlbHRhKGRheXM9aSkgZm9yIGkgaW4gcmFuZ2UoNjApXQpwcmljZXMgPSBwbC5EYXRhRnJhbWUoewogICAgInN5bWJvbCI6IFsiQUFBIl0gKiA2MCArIFsiQkJCIl0gKiA2MCwKICAgICJkYXRlIjogX2RheXMgKiAyLAogICAgImNsb3NlIjogbnAucm91bmQobnAuY29uY2F0ZW5hdGUoWzUwICogbnAuZXhwKG5wLmN1bXN1bShfcm5nLm5vcm1hbCgwLCAwLjAyLCA2MCkpKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICA4MCAqIG5wLmV4cChucC5jdW1zdW0oX3JuZy5ub3JtYWwoMCwgMC4wMiwgNjApKSldKSwgMiksCn0pClBBVEggPSAiL3RtcC9vcmRlcnMuY3N2IgpvcmRlcnMud3JpdGVfY3N2KFBBVEgpCmRlbCBfcm5nCg==").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. Revenue by country
*Easy · id `p1`*

<p><code>orders</code> has one row per order: <code>order_id, user_id, date, country, amount</code>.</p>
<p>Return one row per country with columns <code>country</code>, <code>n_orders</code> (number of orders), <code>total</code> (sum of amount) and <code>avg_amount</code> (mean amount), both rounded to 2 decimals. Sort by <code>total</code>, largest first.</p>
<p class=hint>Use <code>group_by</code> + <code>agg</code> with named expressions.</p>

In [ ]:
def solve(orders: pl.DataFrame) -> pl.DataFrame:
    # your code here
    pass

# Sample run
try:
    print(orders.head())
    print(solve(orders))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p1")

---
## 2. Fill missing ages by group
*Easy · id `p2`*

<p><code>users</code> has <code>user_id, age, country</code>, with some nulls in <code>age</code> and <code>country</code>.</p>
<p>Drop users whose <code>country</code> is null. Fill each missing <code>age</code> with the <b>median age of that user's country</b>, rounded to the nearest integer. Return all three columns sorted by <code>user_id</code>.</p>
<p class=hint><code>fill_null</code> accepts an expression, and <code>.over("country")</code> computes it per group.</p>

In [ ]:
def solve(users: pl.DataFrame) -> pl.DataFrame:
    # your code here
    pass

# Sample run
try:
    print(users.null_count())
    print(solve(users))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p2")

---
## 3. Z-score every float column
*Medium · id `p3`*

<p><code>features</code> has an integer <code>id</code>, several float columns, an integer <code>trades</code> column and a string <code>venue</code>.</p>
<p>Keep every original column and add, for <b>each Float64 column only</b>, a column named <code>&lt;name&gt;_z</code> = (x − mean) / std, using Polars' default sample std. Your code must not name the float columns: if a new float column were added, it should get a z-score too.</p>
<p class=hint><code>pl.col(pl.Float64)</code> or <code>cs.float()</code>, then <code>.name.suffix("_z")</code>.</p>

In [ ]:
def solve(features: pl.DataFrame) -> pl.DataFrame:
    # your code here
    pass

# Sample run
try:
    print(solve(features).head())
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p3")

---
## 4. Audit a join
*Medium · id `p4`*

<p>Before joining <code>orders</code> to <code>users</code>, check the keys.</p>
<p>Return a tuple <code>(orphans, inactive)</code>:</p>
<ul><li><code>orphans</code>: the orders whose <code>user_id</code> has no row in <code>users</code>, all order columns, sorted by <code>order_id</code>.</li>
<li><code>inactive</code>: the users who never placed an order, all user columns, sorted by <code>user_id</code>.</li></ul>
<p class=hint>Look at <code>how="anti"</code> and <code>how="semi"</code>.</p>

In [ ]:
def solve(orders: pl.DataFrame, users: pl.DataFrame):
    # your code here
    pass

# Sample run
try:
    o, i = solve(orders, users)
    print(o)
    print(i)
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p4")

---
## 5. Pivot daily revenue
*Medium · id `p5`*

<p>From <code>orders</code>, build a wide table of daily revenue: one row per <code>date</code> (sorted ascending), then one column per country in alphabetical order (<code>CH, DE, FR, UK</code>). Each cell is that day's summed <code>amount</code>, rounded to 2 decimals, with <b>0</b> where a country had no orders.</p>
<p>Bonus (not tested): <code>unpivot</code> it back to long form and check you get the same totals.</p>
<p class=hint>Aggregate first, then <code>pivot(on=..., index=..., values=...)</code>.</p>

In [ ]:
def solve(orders: pl.DataFrame) -> pl.DataFrame:
    # your code here
    pass

# Sample run
try:
    print(solve(orders))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p5")

---
## 6. Running spend per user
*Medium · id `p6`*

<p>Sort <code>orders</code> by <code>user_id, date, order_id</code>, keep all columns, and add:</p>
<ul><li><code>user_total</code>: the user's total spend across all their orders,</li>
<li><code>cum_spend</code>: the user's running total up to and including this order,</li>
<li><code>share</code>: <code>amount / user_total</code>, rounded to 4 decimals.</li></ul>
<p class=hint><code>.over("user_id")</code> on <code>sum()</code> and on <code>cum_sum()</code>. No joins needed.</p>

In [ ]:
def solve(orders: pl.DataFrame) -> pl.DataFrame:
    # your code here
    pass

# Sample run
try:
    print(solve(orders).head(8))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p6")

---
## 7. 3-day rolling revenue
*Medium · id `p7`*

<p>Compute daily revenue per country from <code>orders</code>: columns <code>country, date, revenue</code>, one row per country and date that has orders, sorted by <code>country, date</code>.</p>
<p>Add <code>rev_3d</code>, the mean of <code>revenue</code> over the current and previous two <b>rows</b> of the same country. It is null for each country's first two rows.</p>
<p class=hint><code>rolling_mean(3)</code> combined with <code>.over("country")</code>.</p>

In [ ]:
def solve(orders: pl.DataFrame) -> pl.DataFrame:
    # your code here
    pass

# Sample run
try:
    print(solve(orders).head(8))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p7")

---
## 8. Ticks to 1-minute OHLC bars
*Hard · id `p8`*

<p><code>ticks</code> is an hour of trades on one instrument: <code>ts</code> (datetime), <code>price</code>, <code>size</code>, sorted by time.</p>
<p>Resample to 1-minute bars. Return <code>ts</code> (the start of the minute), <code>open, high, low, close</code> (first, max, min, last price) and <code>volume</code> (summed size). Only minutes that contain ticks.</p>
<p class=hint><code>group_by_dynamic("ts", every="1m")</code>.</p>

In [ ]:
def solve(ticks: pl.DataFrame) -> pl.DataFrame:
    # your code here
    pass

# Sample run
try:
    print(ticks.head())
    print(solve(ticks).head())
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p8")

---
## 9. Match trades to the prevailing quote
*Hard · id `p9`*

<p><code>trades</code> (<code>ts, symbol, price, size</code>) and <code>quotes</code> (<code>ts, symbol, bid, ask</code>) cover two symbols.</p>
<p>For each trade attach the most recent quote of the <b>same symbol</b> at or before the trade's time. Add <code>mid</code> = (bid + ask) / 2 and <code>buyer_initiated</code> = <code>price &gt;= mid</code>. Return columns <code>ts, symbol, price, size, bid, ask, mid, buyer_initiated</code>, sorted by <code>symbol, ts</code>. A trade before the first quote keeps nulls.</p>
<p class=hint>Both frames must be sorted on the <code>on</code> key.</p>

In [ ]:
def solve(trades: pl.DataFrame, quotes: pl.DataFrame) -> pl.DataFrame:
    # your code here
    pass

# Sample run
try:
    print(solve(trades, quotes).head())
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p9")

---
## 10. Fix the look-ahead leak
*Hard · id `p10`*

<p>A colleague's backtest on <code>prices</code> (<code>symbol, date, close</code>) looks too good. The starter code is their version.</p>
<p>The signal on day <i>t</i> uses day <i>t</i>'s close, then earns day <i>t</i>'s return, which you couldn't have traded. Fix it so <code>strategy_ret</code> uses the <b>previous day's signal of the same symbol</b>. Keep the columns, row order (<code>symbol, date</code>) and the definitions of <code>ret</code> and <code>signal</code> as they are.</p>
<p class=hint><code>shift(1)</code>, and think about which symbol's row it pulls from.</p>

In [ ]:
def solve(prices: pl.DataFrame) -> pl.DataFrame:
    return (
        prices.sort("symbol", "date")
        .with_columns(
            ret=pl.col("close").pct_change().over("symbol"),
            # +1 above the 5-day mean, -1 below, null while the mean is undefined
            signal=(pl.col("close") > pl.col("close").rolling_mean(5).over("symbol")).cast(pl.Int8) * 2 - 1,
        )
        .with_columns(strategy_ret=pl.col("signal") * pl.col("ret"))
        .select("symbol", "date", "ret", "signal", "strategy_ret")
    )

# Sample run
try:
    r = solve(prices)
    print(r.head(8))
    print(r.group_by('symbol').agg(pl.col('strategy_ret').sum()))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p10")

---
## 11. Go lazy: scan, filter, explain
*Hard · id `p11`*

<p>The orders are also saved as a CSV at <code>PATH</code>. Write <code>solve(path)</code> that returns a <b>LazyFrame</b> (don't call <code>collect</code>) built with <code>pl.scan_csv(path)</code> which keeps orders with <code>amount &gt; 20</code>, then gives per country <code>n_orders</code> and <code>total</code> (rounded to 2), sorted by <code>total</code> descending.</p>
<p>Press Run to print <code>explain()</code>. Find where the filter ended up in the plan.</p>
<p class=hint>Build the whole chain on the result of <code>scan_csv</code>; <code>explain()</code> shows the optimised plan, so look for the filter inside the scan.</p>

In [ ]:
def solve(path: str) -> pl.LazyFrame:
    # your code here
    pass

# Sample run
try:
    q = solve(PATH)
    print(q.explain())
    print(q.collect())
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("p11")